# 01 — EDA (CP1)
Flight summary, example flights, hover thrust fit. All logic is in `src/data/`; this notebook only calls it.
Run `pytest tests/test_frames.py -v -s` first — the frame decisions it checks are used here.

Outputs: `results/flight_summary.csv`, `results/hover_thrust_fit.json`, `results/example_flights.json`,
`results/figures/eda_example_flights.png`, `results/figures/eda_hover_thrust.png`.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from src.data.load import segment_index, load_all
from src.data import frames as F, eda
print("quat convention:", F.QUAT_CONVENTION, "| acc is specific force:", F.ACC_IS_SPECIFIC_FORCE)
eda.FIGS.mkdir(parents=True, exist_ok=True)

In [ ]:
idx = segment_index()
df = eda.add_derived(load_all())          # first run parses all CSVs and caches to data/cache/
print(f"{len(idx)} segments, {idx.flight_id.nunique()} flights, {len(df):,} samples = {len(df)*eda.DT/60:.1f} min")

In [ ]:
summary = eda.flight_summary(df, idx)
summary.to_csv("results/flight_summary.csv", index=False, float_format="%.4f")
summary.groupby("trajectory_type").agg(flights=("flight_id","size"), minutes=("duration_s", lambda s: s.sum()/60),
                                       max_speed=("max_speed","max"), max_acc=("max_acc_kin","max")).round(2)

In [ ]:
picks = eda.pick_example_flights(summary)
eda.save_json(picks, "example_flights.json")
print(picks)
eda.plot_example_flights(df, summary, picks, path="results/figures/eda_example_flights.png");

In [ ]:
fit, pts = eda.hover_thrust_fit(df, idx)
eda.save_json(fit, "hover_thrust_fit.json")
print({k: v for k, v in fit.items() if k not in ("mask", "excluded")})
eda.plot_hover_fit(pts, fit, path="results/figures/eda_hover_thrust.png");